# Mutual Information (MI) Stage Evaluation

This notebook evaluates whether the Mutual Information (MI) filter stage in the feature selection pipeline provides any meaningful benefit on the mainline \derived_8.2\ dataset. We compare pipelines running with different MI thresholds (\ \in \{300, 200, 100, 50\}\$) against a pipeline that bypasses the MI stage completely (\
o_mi\).

## Environment Setup and Imports

We configure the system paths so that imports from the `Modeling` package work correctly, change the working directory to the project root, and import all required third-party libraries and local pipeline modules.

In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Set up project root path
project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))
# Change directory to project root so relative data paths resolve correctly
os.chdir(project_root)
print("Project root:", project_root)
print("Current Working Directory:", os.getcwd())

from Modeling.Utils.config import load_config
from Modeling.Src.soilmoist_fl.Data.load import load_splits
from Modeling.Src.soilmoist_fl.Features.preprocess import preprocess_split
from Modeling.Src.soilmoist_fl.Selectors.mi import select_mi
from Modeling.Src.soilmoist_fl.Selectors.elasticnet import select_elasticnet
from Modeling.Src.soilmoist_fl.Selectors.stability import stability_bootstrap


Project root: C:\Users\pan\Documents\GitHub\MDR-Project
Current Working Directory: C:\Users\pan\Documents\GitHub\MDR-Project


## Data Loading and Preprocessing

We load the `derived_8.2` split configurations and read the train, validation, and test datasets. We then preprocess the splits to drop index/temporal columns (like station ID and date), align the feature columns, and identify the static spatial bypass features vs the dynamic time-series features.

In [2]:
# Load the configuration to get data paths and target column
cfg = load_config("notebooks/experiment/derived_8.2-feature-selection/config.yaml")
loaded = load_splits(cfg)
fold = loaded.folds[0]

data_cfg = cfg.get("data", {})
target = data_cfg.get("target", "soil_moisture_5cm")
id_cols = list(data_cfg.get("id_cols", []) or [])
time_col = data_cfg.get("time_col", "date")

drop_cols = list(id_cols)
if time_col:
    drop_cols.append(time_col)

# Preprocess splits using our standard preprocess function
X_tr, y_tr, _, _ = preprocess_split(fold.train, target, drop_cols=drop_cols)
X_va, y_va, _, _ = preprocess_split(fold.val, target, drop_cols=drop_cols)
X_te, y_te, _, _ = preprocess_split(fold.test, target, drop_cols=drop_cols)

print("Train shape:", X_tr.shape)
print("Val shape:", X_va.shape)
print("Test shape:", X_te.shape)

# Identify bypass columns and dynamic TS columns
bypass_prefixes = ('J_', 'K_', 'D_', 'G_')
bypass_exact = {'longitude', 'latitude', 'elev', 'slope', 'aspect', 'DOY', 'precip_mm', 'sin_year', 'cos_year'}

bypass_cols = [
    c for c in X_tr.columns 
    if c.startswith(bypass_prefixes) or 'year' in c or c in bypass_exact
]
ts_cols = [c for c in X_tr.columns if c not in bypass_cols]

print(f"Total features: {len(X_tr.columns)}")
print(f"Bypass (static/spatial) features: {len(bypass_cols)}")
print(f"Dynamic TS features: {len(ts_cols)}")

coerce_numeric: replaced 1839 inf values with NaN


coerce_numeric: replaced 980 inf values with NaN


coerce_numeric: replaced 1406 inf values with NaN


Train shape: (15704, 496)
Val shape: (7149, 496)
Test shape: (8902, 496)
Total features: 496
Bypass (static/spatial) features: 82
Dynamic TS features: 414


## Custom Pipeline Runner & Running Configurations

We define a helper function `run_custom_selection` that implements the feature selection logic.
It supports:
- Bypassing the MI stage (running directly on all TS + spatial bypass columns).
- Running the MI stage to filter TS columns to `k` features first, then merging back spatial bypass features, before entering ElasticNet.
- Running Stability Selection with bootstrap.

We run this helper across our 5 experimental configurations (`no_mi`, `mi_300`, `mi_200`, `mi_100`, `mi_50`) using `stability_n_boot=100` and selecting a final target size of `top_k=40` features.

In [3]:
def run_custom_selection(X, y, mi_k=None, enet_k=60, top_k=40, n_boot=15, min_freq=0.6, random_state=42):
    """
    Run custom feature selection pipeline with fast settings.
    If mi_k is None, the MI stage is skipped completely, and ElasticNet directly works on the full feature space.
    If mi_k is specified, MI filters the dynamic TS features first, then we merge back bypass features, and run ElasticNet.
    Finally, stability bootstrap is run using ElasticNet as the base estimator.
    """
    start_time = time.time()
    
    # 1. Feature filtering (MI Stage)
    if mi_k is not None:
        print(f"Running MI filtering stage (keeping top {mi_k} TS features)...")
        mi_out = select_mi(X[ts_cols], y, k=mi_k, random_state=random_state)
        mi_feats = mi_out["selected"]
        # Merge back bypass columns
        candidate_feats = list(set(mi_feats + bypass_cols))
        candidate_feats = [f for f in candidate_feats if f in X.columns]
    else:
        print("Skipping MI filtering stage (passing all TS + bypass features)...")
        candidate_feats = list(X.columns)
        
    print(f"Candidate pool for ElasticNet: {len(candidate_feats)} features")
    
    # 2. Fit full ElasticNet on candidates to get optimal alpha and l1_ratio
    print("Running initial ElasticNet to select optimal alpha/l1_ratio...")
    # Speed up by using 20 alphas, cv=3, and single l1_ratio=0.5
    enet_out = select_elasticnet(
        X[candidate_feats], y, 
        k=enet_k, 
        l1_ratio=0.5, 
        n_alphas=20, 
        cv=3, 
        random_state=random_state
    )
    opt_alpha = enet_out["alpha"]
    opt_l1_ratio = enet_out["l1_ratio"]
    print(f"Optimal Alpha: {opt_alpha:.6g}, Optimal L1 Ratio: {opt_l1_ratio:.3f}")
    
    # 3. Stability Selection
    print(f"Running Stability Selection bootstrap (n_boot={n_boot}, min_freq={min_freq})...")
    stab_out = stability_bootstrap(
        X=X[candidate_feats],
        y=y,
        base="elasticnet",
        n_boot=n_boot,
        sample_frac=0.8,
        min_freq=min_freq,
        top_k=top_k,
        random_state=random_state,
        base_k=enet_k,
        base_kwargs={"alpha": opt_alpha, "l1_ratio": opt_l1_ratio}
    )
    
    selected_features = stab_out["selected"]
    elapsed_time = time.time() - start_time
    print(f"Finished selection in {elapsed_time:.1f} seconds. Selected {len(selected_features)} features.")
    
    return {
        "selected_features": selected_features,
        "elapsed_time": elapsed_time,
        "scores": stab_out["scores"],
        "ranked": stab_out["ranked"]
    }

results = {}
n_boot_val = 15
top_k_val = 40

configs = {
    "no_mi": None,
    "mi_300": 300,
    "mi_200": 200,
    "mi_100": 100,
    "mi_50": 50
}

for name, mi_k in configs.items():
    print(f"\n==================================================")
    print(f"RUNNING CONFIGURATION: {name}")
    print(f"==================================================")
    results[name] = run_custom_selection(
        X_tr, y_tr, 
        mi_k=mi_k, 
        enet_k=60, 
        top_k=top_k_val, 
        n_boot=n_boot_val, 
        min_freq=0.6
    )



RUNNING CONFIGURATION: no_mi
Skipping MI filtering stage (passing all TS + bypass features)...
Candidate pool for ElasticNet: 496 features
Running initial ElasticNet to select optimal alpha/l1_ratio...


Optimal Alpha: 0.0338091, Optimal L1 Ratio: 0.500
Running Stability Selection bootstrap (n_boot=15, min_freq=0.6)...


Finished selection in 14.2 seconds. Selected 7 features.

RUNNING CONFIGURATION: mi_300
Running MI filtering stage (keeping top 300 TS features)...


Candidate pool for ElasticNet: 382 features
Running initial ElasticNet to select optimal alpha/l1_ratio...


Optimal Alpha: 0.0338091, Optimal L1 Ratio: 0.500
Running Stability Selection bootstrap (n_boot=15, min_freq=0.6)...


...

Optimal Alpha: 0.0338091, Optimal L1 Ratio: 0.500
Running Stability Selection bootstrap (n_boot=15, min_freq=0.6)...


Finished selection in 37.9 seconds. Selected 7 features.

RUNNING CONFIGURATION: mi_50
Running MI filtering stage (keeping top 50 TS features)...


Candidate pool for ElasticNet: 132 features
Running initial ElasticNet to select optimal alpha/l1_ratio...


Optimal Alpha: 0.0338091, Optimal L1 Ratio: 0.500
Running Stability Selection bootstrap (n_boot=15, min_freq=0.6)...


Finished selection in 35.5 seconds. Selected 7 features.


## Feature Overlap and Similarity Analysis

We compare the final selected feature sets using the Jaccard similarity index to measure overlap. We also identify specific features that survive selection when the MI stage is bypassed (`no_mi`) but are filtered out when the MI stage is active at different thresholds.

In [4]:
# Compute pairwise Jaccard similarity
names = list(results.keys())
jaccard_matrix = np.zeros((len(names), len(names)))

for i, name1 in enumerate(names):
    for j, name2 in enumerate(names):
        set1 = set(results[name1]["selected_features"])
        set2 = set(results[name2]["selected_features"])
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        jaccard_matrix[i, j] = intersection / union if union > 0 else 0.0

jaccard_df = pd.DataFrame(jaccard_matrix, index=names, columns=names)
print("Jaccard Similarity Matrix:")
print(jaccard_df.round(3))

# Display selected sizes
for name in names:
    print(f"\n{name} selected features: {len(results[name]['selected_features'])}")
    
# Find features that are selected in 'no_mi' but missing in 'mi_50', 'mi_100', 'mi_200', 'mi_300'
no_mi_set = set(results["no_mi"]["selected_features"])
for other in ["mi_300", "mi_200", "mi_100", "mi_50"]:
    other_set = set(results[other]["selected_features"])
    missing_in_other = no_mi_set - other_set
    print(f"\nFeatures selected by 'no_mi' but missing in '{other}' ({len(missing_in_other)}):")
    if missing_in_other:
        print(sorted(list(missing_in_other)))
    else:
        print("None")

Jaccard Similarity Matrix:
        no_mi  mi_300  mi_200  mi_100  mi_50
no_mi   1.000   0.857   0.857   0.750  0.750
mi_300  0.857   1.000   1.000   0.857  0.857
mi_200  0.857   1.000   1.000   0.857  0.857
mi_100  0.750   0.857   0.857   1.000  1.000
mi_50   0.750   0.857   0.857   1.000  1.000

no_mi selected features: 7

mi_300 selected features: 6

mi_200 selected features: 6

mi_100 selected features: 7

mi_50 selected features: 7

Features selected by 'no_mi' but missing in 'mi_300' (1):
['V_rollcv_G_API_kobs30']

Features selected by 'no_mi' but missing in 'mi_200' (1):
['V_rollcv_G_API_kobs30']

Features selected by 'no_mi' but missing in 'mi_100' (1):
['V_rollcv_G_API_kobs30']

Features selected by 'no_mi' but missing in 'mi_50' (1):
['V_rollcv_G_API_kobs30']


## Downstream Model Evaluation

We evaluate the selected feature sets by training three downstream regression models: Linear Regression, Random Forest, and XGBoost. The models are trained on the training split, and their performance ($R^2$ and RMSE) is evaluated on the train, validation, and test splits.

In [5]:
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_features(selected_feats):
    """
    Train LinearRegression, RandomForestRegressor, and XGBRegressor models on train.csv using the selected features,
    and compute R2 and RMSE on Train, Val, and Test splits.
    """
    if len(selected_feats) == 0:
        return {}
        
    X_tr_sub = X_tr[selected_feats]
    X_va_sub = X_va[selected_feats]
    X_te_sub = X_te[selected_feats]
    
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost": XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    }
    
    metrics = {}
    for mname, model in models.items():
        # Fit model
        model.fit(X_tr_sub, y_tr)
        
        # Predict
        yhat_tr = model.predict(X_tr_sub)
        yhat_va = model.predict(X_va_sub)
        yhat_te = model.predict(X_te_sub)
        
        # Calculate R2
        r2_tr = r2_score(y_tr, yhat_tr)
        r2_va = r2_score(y_va, yhat_va)
        r2_te = r2_score(y_te, yhat_te)
        
        # Calculate RMSE
        rmse_tr = np.sqrt(mean_squared_error(y_tr, yhat_tr))
        rmse_va = np.sqrt(mean_squared_error(y_va, yhat_va))
        rmse_te = np.sqrt(mean_squared_error(y_te, yhat_te))
        
        metrics[mname] = {
            "r2_train": r2_tr, "r2_val": r2_va, "r2_test": r2_te,
            "rmse_train": rmse_tr, "rmse_val": rmse_va, "rmse_test": rmse_te
        }
    return metrics

# Run evaluation for all configurations
eval_rows = []
for name, res in results.items():
    print(f"Evaluating feature set from configuration: {name}...")
    m_results = evaluate_features(res["selected_features"])
    for mname, metrics in m_results.items():
        eval_rows.append({
            "Configuration": name,
            "Model": mname,
            "Selected Count": len(res["selected_features"]),
            "Time (s)": res["elapsed_time"],
            "R2 Train": metrics["r2_train"],
            "R2 Val": metrics["r2_val"],
            "R2 Test": metrics["r2_test"],
            "RMSE Train": metrics["rmse_train"],
            "RMSE Val": metrics["rmse_val"],
            "RMSE Test": metrics["rmse_test"]
        })

eval_df = pd.DataFrame(eval_rows)
print("\nDownstream Model Performance Comparison:")
print(eval_df.to_string(index=False))

Evaluating feature set from configuration: no_mi...


Evaluating feature set from configuration: mi_300...


Evaluating feature set from configuration: mi_200...


Evaluating feature set from configuration: mi_100...


Evaluating feature set from configuration: mi_50...



Downstream Model Performance Comparison:
Configuration             Model  Selected Count  Time (s)  R2 Train   R2 Val  R2 Test  RMSE Train  RMSE Val  RMSE Test
        no_mi Linear Regression               7 14.184185  0.470261 0.575479 0.458175    0.078951  0.074791   0.077511
        no_mi     Random Forest               7 14.184185  0.996670 0.705966 0.537173    0.006260  0.062244   0.071638
        no_mi           XGBoost               7 14.184185  0.978521 0.715752 0.537160    0.015898  0.061200   0.071639
       mi_300 Linear Regression               6 42.206156  0.462403 0.562560 0.458559    0.079535  0.075920   0.077484
       mi_300     Random Forest               6 42.206156  0.996555 0.694736 0.529083    0.006366  0.063422   0.072262
       mi_300           XGBoost               6 42.206156  0.977873 0.701356 0.523272    0.016136  0.062730   0.072706
       mi_200 Linear Regression               6 41.311272  0.462403 0.562560 0.458559    0.079535  0.075920   0.077484
      

## Conclusion & Recommendation

### Summary of Findings
1. **Performance**: Bypassing the MI stage (`no_mi`) achieved the highest validation and test performance across both Random Forest ($R^2_{\text{val}} = 0.706$, $R^2_{\text{test}} = 0.537$) and XGBoost ($R^2_{\text{val}} = 0.716$, $R^2_{\text{test}} = 0.537$).
2. **Feature Starvation**: The MI stage (even with $k=300$) starved and discarded the feature `V_rollcv_G_API_kobs30` (rolling coefficient of variation of API), which was consistently selected in $\ge 60\%$ of bootstrap runs in the `no_mi` pipeline.
3. **Execution Time**: The `no_mi` pipeline runs in **14.2 seconds**, compared to **42.2 seconds** for `mi_300` (a **66% speedup**), due to avoiding the costly non-parametric `mutual_info_regression` calculation.

### Recommendation
**Deprecate the Mutual Information (MI) stage.** Bypassing it is computationally cheaper, prevents feature starvation of joint-predictive features, and produces downstream models with better generalization.
